# Baseline: Random Forest for Unit Commitment Prediction

A minimal, "as simple as possible" baseline for the Tokke-Vinje unit commitment task:

1. Build one feature vector per weekly case (`Run No` / `starttime`) from the day-ahead price,
   inflow, reservoir volume/water-value, and the min-flow/min-volume constraints.
2. Fit a single multi-output `RandomForestClassifier` that predicts all 2,352
   `result_committed_<Plant>_G<n>_t<0-167>` columns at once.
3. Validate with a time-based train/validation split and score with the competition's
   micro-average ROC AUC (`kaggle_metric.score`).
4. Use the fitted model to produce a `sample_submission`-shaped file for the held-out
   `prediction_mapping.csv` cases (2023-01-01 to 2024-12-25), which have no ground truth here.

This is intentionally simple: features are coarse (full-week hourly price, weekly-mean inflow,
start-of-week volume/water value/constraints) and the model is a single off-the-shelf
`RandomForestClassifier`. It is meant as a working starting point, not a tuned solution.

In [1]:
import time
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.ensemble import RandomForestClassifier

from kaggle_metric import score as kaggle_score

HORIZON = 168  # hours in one weekly case

## 1. Load data

In [2]:
BASE_PATH = Path(".")
KERNEL_PATH = BASE_PATH / "data" / "kernel"
EXTENDED_PATH = BASE_PATH / "data" / "extended"


def load_series(path, date_col="date"):
    df = pd.read_csv(path, parse_dates=[date_col])
    df[date_col] = df[date_col].dt.tz_localize(None)
    return df


price = load_series(KERNEL_PATH / "Historical_day_ahead_price_2015_2025.csv")
inflow = load_series(KERNEL_PATH / "Historical_inflow_1958_2025.csv")
volume = load_series(KERNEL_PATH / "Historical_volume_2015_2024.csv")
water_value = load_series(KERNEL_PATH / "Synthetic_water_value_2015_2024.csv")
min_flow = load_series(EXTENDED_PATH / "Constraint_min_flow.csv")
min_volume = load_series(EXTENDED_PATH / "Constraint_min_volume.csv")

uc = pd.read_csv(KERNEL_PATH / "Unit_commitment_decisions.csv", parse_dates=["starttime"])
prediction_mapping = pd.read_csv(BASE_PATH / "prediction_mapping.csv", parse_dates=["starttime"])
sample_submission = pd.read_csv(BASE_PATH / "sample_submission.csv")

label_cols = [c for c in uc.columns if c.startswith("result_committed")]
print(f"{len(label_cols)} label columns, {len(uc)} labeled weekly cases, "
      f"{len(prediction_mapping)} cases to predict for submission")

2352 label columns, 2922 labeled weekly cases, 725 cases to predict for submission


## 2. Build features per weekly case

For every case we use:

- **Price** – the day-ahead price for all 168 hours of the week (it is known in advance and is a
  direct driver of the commitment decision).
- **Inflow** – the mean inflow over the week, per reservoir/river node.
- **Volume / water value** – the daily value at the *start* of the week, per reservoir.
- **Min-flow / min-volume constraints** – the value at the start of the week.

All lookups use `np.searchsorted` on the (sorted, regularly spaced) hourly/daily date arrays, so a
whole feature matrix can be built with array indexing instead of a per-row loop.

In [3]:
def series_arrays(df, date_col="date"):
    value_cols = [c for c in df.columns if c != date_col]
    return df[date_col].to_numpy(), df[value_cols].to_numpy(dtype=float), value_cols


price_dates, price_values, _ = series_arrays(price)
inflow_dates, inflow_values, inflow_cols = series_arrays(inflow)
volume_dates, volume_values, volume_cols = series_arrays(volume)
water_value_dates, water_value_values, water_value_cols = series_arrays(water_value)
min_flow_dates, min_flow_values, min_flow_cols = series_arrays(min_flow)
min_volume_dates, min_volume_values, min_volume_cols = series_arrays(min_volume)

feature_names = (
    [f"price_t{t}" for t in range(HORIZON)]
    + [f"inflow_mean_{c}" for c in inflow_cols]
    + [f"volume_start_{c}" for c in volume_cols]
    + [f"water_value_start_{c}" for c in water_value_cols]
    + [f"min_flow_start_{c}" for c in min_flow_cols]
    + [f"min_volume_start_{c}" for c in min_volume_cols]
)


def build_features(starttimes):
    starttimes = pd.to_datetime(starttimes).to_numpy()

    price_pos = np.searchsorted(price_dates, starttimes)
    assert np.all(price_dates[price_pos] == starttimes), "starttime not aligned with price grid"
    price_idx = price_pos[:, None] + np.arange(HORIZON)[None, :]
    price_feat = price_values[price_idx, 0]

    inflow_pos = np.searchsorted(inflow_dates, starttimes)
    inflow_idx = inflow_pos[:, None] + np.arange(HORIZON)[None, :]
    inflow_feat = inflow_values[inflow_idx].mean(axis=1)

    volume_feat = volume_values[np.searchsorted(volume_dates, starttimes)]
    water_value_feat = water_value_values[np.searchsorted(water_value_dates, starttimes)]
    min_flow_feat = min_flow_values[np.searchsorted(min_flow_dates, starttimes)]
    min_volume_feat = min_volume_values[np.searchsorted(min_volume_dates, starttimes)]

    return np.hstack(
        [price_feat, inflow_feat, volume_feat, water_value_feat, min_flow_feat, min_volume_feat]
    )


X_all = build_features(uc["starttime"])
Y_all = uc[label_cols].to_numpy()
print("X_all shape:", X_all.shape, "Y_all shape:", Y_all.shape)

X_all shape: (2922, 229) Y_all shape: (2922, 2352)


## 3. Time-based train / validation split

The weekly cases are daily rolling windows, so consecutive cases overlap heavily. A random
split would leak almost-identical windows between train and validation, so instead the most
recent ~15% of cases (by `starttime`) are held out.

In [4]:
uc_sorted = uc.sort_values("starttime").reset_index(drop=True)
X_all = build_features(uc_sorted["starttime"])
Y_all = uc_sorted[label_cols].to_numpy()

n_val = int(0.15 * len(uc_sorted))
X_train, X_val = X_all[:-n_val], X_all[-n_val:]
Y_train, Y_val = Y_all[:-n_val], Y_all[-n_val:]

print(f"train: {X_train.shape[0]} cases ({uc_sorted['starttime'].iloc[0].date()} to "
      f"{uc_sorted['starttime'].iloc[-n_val - 1].date()})")
print(f"val:   {X_val.shape[0]} cases ({uc_sorted['starttime'].iloc[-n_val].date()} to "
      f"{uc_sorted['starttime'].iloc[-1].date()})")

train: 2484 cases (2015-01-01 to 2021-10-19)
val:   438 cases (2021-10-20 to 2022-12-31)


## 4. Fit a single multi-output Random Forest

`RandomForestClassifier` supports multi-output targets directly: `Y_train` has one column per
`result_committed_*` label, and a single model predicts all of them jointly.

In [5]:
model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)

start = time.time()
model.fit(X_train, Y_train)
print(f"fit in {time.time() - start:.1f}s")

fit in 68.0s


## 5. Predict probabilities and score

`predict_proba` on a multi-output classifier returns one array per label. A label that is
constant in the training split only has one class, so we fall back to a constant probability
for those columns instead of indexing a missing class.

In [6]:
def predict_proba_matrix(model, X):
    proba_per_label = model.predict_proba(X)
    n_samples = X.shape[0]
    n_labels = len(proba_per_label)
    out = np.empty((n_samples, n_labels))
    for i, (classes, proba) in enumerate(zip(model.classes_, proba_per_label)):
        if 1 in classes:
            out[:, i] = proba[:, list(classes).index(1)]
        else:
            out[:, i] = 0.0  # label was always 0 in the training split
    return out


val_proba = predict_proba_matrix(model, X_val)

solution_df = pd.DataFrame(Y_val, columns=label_cols)
solution_df.insert(0, "Run No", uc_sorted["Run No"].iloc[-n_val:].to_numpy())

submission_df = pd.DataFrame(val_proba, columns=label_cols)
submission_df.insert(0, "Run No", uc_sorted["Run No"].iloc[-n_val:].to_numpy())

val_auc = kaggle_score(solution_df, submission_df)
print(f"Validation micro-average ROC AUC: {val_auc:.4f}")

Validation micro-average ROC AUC: 0.6198


## 6. Which features matter most?

A quick look at the Random Forest's built-in feature importances (averaged across all 2,352
outputs by the ensemble).

In [7]:
importances = pd.Series(model.feature_importances_, index=feature_names)
importances.sort_values(ascending=False).head(20)

water_value_start_Bitdalsvatn      0.022803
water_value_start_Songavatn        0.021637
water_value_start_Vaamarvatn       0.021275
water_value_start_Totak            0.020503
water_value_start_Botnedalsvatn    0.020477
water_value_start_Hyljelihyl       0.020174
volume_start_Bordalsvatn           0.019796
water_value_start_Staavatn         0.019626
water_value_start_Venemo           0.019234
water_value_start_Kjelavatn        0.018352
volume_start_Langesae              0.017381
volume_start_Bitdalsvatn           0.017156
volume_start_Kjelavatn             0.016997
water_value_start_Bordalsvatn      0.015624
water_value_start_Vatjern          0.014869
water_value_start_Foersvatn        0.014397
volume_start_Songavatn             0.013773
volume_start_Foersvatn             0.013626
water_value_start_Langesae         0.013597
water_value_start_Byrtevatn        0.012653
dtype: float64

## 7. Build a submission for the unlabeled prediction cases

`prediction_mapping.csv` lists the `Run No` / `starttime` cases (2023-01-01 to 2024-12-25) that
have no ground truth in this dataset. We refit on *all* labeled cases (train + validation) and
predict probabilities for those cases in the `sample_submission.csv` format.

In [8]:
final_model = RandomForestClassifier(
    n_estimators=200,
    max_depth=12,
    min_samples_leaf=5,
    n_jobs=-1,
    random_state=42,
)
final_model.fit(X_all, Y_all)

X_submit = build_features(prediction_mapping["starttime"])
submit_proba = predict_proba_matrix(final_model, X_submit)

submission = pd.DataFrame(submit_proba, columns=label_cols)
submission.insert(0, "Run No", prediction_mapping["Run No"].to_numpy())
submission = submission[sample_submission.columns]  # match exact column order

submission.to_csv("random_forest_submission.csv", index=False)
submission.head()

,Run No,result_committed_Hogga_G1_t0,result_committed_Hogga_G1_t1,result_committed_Hogga_G1_t2,result_committed_Hogga_G1_t3,result_committed_Hogga_G1_t4,result_committed_Hogga_G1_t5,result_committed_Hogga_G1_t6,result_committed_Hogga_G1_t7,result_committed_Hogga_G1_t8,...,result_committed_Vesle Kjela_G1_t158,result_committed_Vesle Kjela_G1_t159,result_committed_Vesle Kjela_G1_t160,result_committed_Vesle Kjela_G1_t161,result_committed_Vesle Kjela_G1_t162,result_committed_Vesle Kjela_G1_t163,result_committed_Vesle Kjela_G1_t164,result_committed_Vesle Kjela_G1_t165,result_committed_Vesle Kjela_G1_t166,result_committed_Vesle Kjela_G1_t167
0,2923,0.221365,0.221365,0.223605,0.224676,0.300957,0.375308,0.423744,0.447838,0.520451,...,0.805222,0.809786,0.807293,0.807030,0.797442,0.791420,0.784448,0.783084,0.780564,0.780564
1,2924,0.269512,0.269512,0.277704,0.284238,0.360198,0.427420,0.479261,0.517159,0.569845,...,0.754720,0.761788,0.760052,0.758677,0.751720,0.744722,0.739644,0.739644,0.737485,0.737485
2,2925,0.278469,0.278469,0.287723,0.288794,0.361440,0.433066,0.473805,0.501879,0.561390,...,0.742113,0.752872,0.752455,0.748988,0.735207,0.728021,0.723943,0.723943,0.721479,0.721479
3,2926,0.258796,0.258796,0.267957,0.274771,0.346817,0.412719,0.456368,0.474969,0.532656,...,0.757583,0.766417,0.761383,0.760286,0.746819,0.739831,0.728985,0.728985,0.722331,0.722331
4,2927,0.287597,0.287597,0.300303,0.305967,0.367838,0.421307,0.470153,0.502065,0.540687,...,0.705837,0.709968,0.708539,0.701953,0.691723,0.681701,0.676076,0.676076,0.674042,0.674042
